In [0]:
%python
import requests 
import json
from pyspark.sql.functions import *
from pyspark.sql.types import * 


In [0]:
%python
spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cricket_api_project")
base_path='/Volumes/workspace/default/cricket_api_project'

In [0]:
%python
API_KEY = "dd4dd816-1fa4-4d7b-9476-06a18b611072"
api_url = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"
response = requests.get(api_url)
response.raise_for_status()
api_data = response.json()
print(api_data.keys())
print(json.dumps(api_data,indent= 2)[:2000])

In [0]:
%python
raw_path_file = f"{base_path}/current_matches_raw.json"
with open(raw_path_file, "w") as file:
    json.dump(api_data, file)
print("Raw API Data is saved at: ", raw_path_file)

In [0]:
%python
bronze_data = [{
    'source_api':api_url,
    'raw_json' : json.dumps(api_data),
    'ingestion_time' : None
}]

bronze_schema = StructType([
    StructField("source_api", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

bronze_df=spark.createDataFrame(bronze_data,bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())
display(bronze_df)


In [0]:
%python
bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("Workspace.default.cricket_bronze_current_matches")

print("Bronze Layer Table Created Successfully")

In [0]:
select * from Workspace.default.cricket_bronze_current_matches